In [0]:
from pyspark.sql import functions as F

# --- Silver: Customers ---
silver_customers = (spark.table("workspace.default.bronze_customers")
    .dropDuplicates(["customer_id"])
    .withColumn("signup_date", F.to_date("signup_date"))
    .select("customer_id", "name", "city", "signup_date")
)
silver_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_customers")

# --- Silver: Products (keep full SCD history, just clean types) ---
silver_products = (spark.table("workspace.default.bronze_products")
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("effective_date", F.to_date("effective_date"))
    .withColumn("end_date", F.to_date("end_date"))
    .select("product_id", "price", "category", "effective_date", "end_date", "is_current")
)
silver_products.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_products")

# --- Silver: Orders ---
silver_orders = (spark.table("workspace.default.bronze_orders")
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("discount", F.coalesce(F.col("discount").cast("double"), F.lit(0.0)))
    .select("order_id", "product_id", "customer_id", "quantity", "price",
            "order_date", "order_status", "channel", "discount")
)
silver_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_orders")

# --- Silver: Payments ---
silver_payments = (spark.table("workspace.default.bronze_payments")
    .withColumn("payment_amount", F.col("payment_amount").cast("double"))
    .withColumn("payment_date", F.to_date("payment_date"))
    .select("order_id", "payment_amount", "payment_method", "payment_date")
)
silver_payments.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_payments")

In [0]:
silver_orders.display()

order_id,product_id,customer_id,quantity,price,order_date,order_status,channel,discount
1,29,1950,4,1701.72,2024-02-29,Cancelled,Web,0.05
2,98,1491,5,1643.01,2024-01-28,Completed,Store,0.16
3,2,1518,4,1948.47,2024-01-10,Completed,Store,0.18
4,33,440,3,1911.69,2024-02-01,Completed,Web,0.07
5,1,1011,4,795.88,2024-02-11,Completed,App,0.25
6,31,741,3,625.85,2024-01-31,Completed,Store,0.15
7,35,201,3,174.15,2024-01-14,Completed,Web,0.22
8,100,490,5,823.93,2024-02-29,Cancelled,Web,0.06
9,39,269,1,1880.09,2024-02-10,Cancelled,Web,0.27
10,50,1203,3,522.32,2024-01-12,Completed,App,0.19


In [0]:
window_count = silver_orders.groupBy("order_id").count()
duplicate_order_ids = window_count.filter(F.col("count") > 1)
print("Duplicate order_id count:", duplicate_order_ids.count())

Duplicate order_id count: 400


In [0]:
silver_orders_priced = (silver_orders.alias("o")
    .join(
        silver_products.alias("p"),
        (F.col("o.product_id") == F.col("p.product_id")) &
        (F.col("o.order_date") >= F.col("p.effective_date")) &
        ((F.col("o.order_date") <= F.col("p.end_date")) | F.col("p.end_date").isNull()),
        how="left"
    )
    .select(
        F.col("o.order_id"), F.col("o.product_id"), F.col("o.customer_id"),
        F.col("o.quantity"), F.col("o.price").alias("order_price"),
        F.col("p.price").alias("catalog_price"),
        F.col("o.order_date"), F.col("o.order_status"), F.col("o.channel"), F.col("o.discount")
    )
)
silver_orders_priced.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_orders_priced")
silver_orders_priced.display()

order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,order_status,channel,discount
1,29,1950,4,1701.72,1136.85,2024-02-29,Cancelled,Web,0.05
2,98,1491,5,1643.01,1658.33,2024-01-28,Completed,Store,0.16
3,2,1518,4,1948.47,1837.12,2024-01-10,Completed,Store,0.18
4,33,440,3,1911.69,1071.5,2024-02-01,Completed,Web,0.07
5,1,1011,4,795.88,1781.59,2024-02-11,Completed,App,0.25
6,31,741,3,625.85,1678.43,2024-01-31,Completed,Store,0.15
7,35,201,3,174.15,1507.39,2024-01-14,Completed,Web,0.22
8,100,490,5,823.93,369.61,2024-02-29,Cancelled,Web,0.06
9,39,269,1,1880.09,278.64,2024-02-10,Cancelled,Web,0.27
10,50,1203,3,522.32,570.28,2024-01-12,Completed,App,0.19


In [0]:
print("silver_orders:", spark.table("workspace.default.silver_orders").count())
print("silver_payments:", spark.table("workspace.default.silver_payments").count())
print("silver_customers:", spark.table("workspace.default.silver_customers").count())
print("silver_products:", spark.table("workspace.default.silver_products").count())
print("silver_orders_priced:", spark.table("workspace.default.silver_orders_priced").count())

silver_orders: 20400
silver_payments: 18600
silver_customers: 2000
silver_products: 200
silver_orders_priced: 20400


In [0]:
spark.table("workspace.default.silver_orders_priced").display()

order_id,product_id,customer_id,quantity,order_price,catalog_price,order_date,order_status,channel,discount
1,29,1950,4,1701.72,1136.85,2024-02-29,Cancelled,Web,0.05
2,98,1491,5,1643.01,1658.33,2024-01-28,Completed,Store,0.16
3,2,1518,4,1948.47,1837.12,2024-01-10,Completed,Store,0.18
4,33,440,3,1911.69,1071.5,2024-02-01,Completed,Web,0.07
5,1,1011,4,795.88,1781.59,2024-02-11,Completed,App,0.25
6,31,741,3,625.85,1678.43,2024-01-31,Completed,Store,0.15
7,35,201,3,174.15,1507.39,2024-01-14,Completed,Web,0.22
8,100,490,5,823.93,369.61,2024-02-29,Cancelled,Web,0.06
9,39,269,1,1880.09,278.64,2024-02-10,Cancelled,Web,0.27
10,50,1203,3,522.32,570.28,2024-01-12,Completed,App,0.19
